# Get the Syzhet for each novel in a series then Map over Narartive time


## Install Packages

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from IPython.display import display, HTML

## Get Data

In [2]:
OHCO = ['title', 'chapter_id', 'paragraph_id', 'sent_id', 'token_id']
SENTS = OHCO[:4]
PARA = OHCO[:3]
CHAP = OHCO[:2]
BOOK = OHCO[:1]
directory_path = 'C:/Users/mason/Box/Text As Data Final'

In [8]:
CORPUS_file = directory_path + '/Output/BrandonSanderson_CORPUS.csv'
LIB_file = directory_path + '/Output/BrandonSanderson_LIB_cosmere.csv'
syuzhet_file = directory_path + '/Lexicon/salex_syuzhet.csv'

In [9]:
SALEX = pd.read_csv(syuzhet_file).set_index('term_str')
SALEX

,syu_sentiment
term_str,
abandon,-0.75
abandoned,-0.50
abandoner,-0.25
abandonment,-0.25
abandons,-1.00
...,...
zest,0.50
zombie,-0.25
zombies,-0.25


## Get LIB and CORPUS

In [11]:
LIB = pd.read_csv(LIB_file).set_index('title').sort_index()
LIB.drop(columns=['Length', 'Total_chapters', 'Total_paragraphs', 'File_path', 'label'], inplace=True)
TOKENS = pd.read_csv(CORPUS_file, index_col=0).set_index(OHCO)
TOKENS.drop(columns=['token_str'], inplace=True)
TOKENS = TOKENS.join(LIB)
TOKENS.head()

term_str  pos  \
title             chapter_id paragraph_id sent_id token_id                 
A Memory of Light 0          0            0       0             the   DT   
                                                  1           wheel  NNP   
                                                  2              of   IN   
                                                  3            time  NNP   
                                                  4           turns  NNS   

                                                           pos_group  \
title             chapter_id paragraph_id sent_id token_id             
A Memory of Light 0          0            0       0               DT   
                                                  1               NN   
                                                  2               IN   
                                                  3               NN   
                                                  4               NN   

                                                                                       Author  \
title             chapter_id paragraph_id sent_id token_id                                      
A Memory of Light 0          0            0       0         Robert Jordan & Brandon Sanderson   
                                                  1         Robert Jordan & Brandon Sanderson   
                                                  2         Robert Jordan & Brandon Sanderson   
                                                  3         Robert Jordan & Brandon Sanderson   
                                                  4         Robert Jordan & Brandon Sanderson   

                                                                  Date  \
title             chapter_id paragraph_id sent_id token_id               
A Memory of Light 0          0            0       0         2013-01-08   
                                                  1         2013-01-08   
                                                  2         2013-01-08   
                                                  3         2013-01-08   
                                                  4         2013-01-08   

                                                                       Series  \
title             chapter_id paragraph_id sent_id token_id                      
A Memory of Light 0          0            0       0         The Wheel of Time   
                                                  1         The Wheel of Time   
                                                  2         The Wheel of Time   
                                                  3         The Wheel of Time   
                                                  4         The Wheel of Time   

                                                                Cosmere  
title             chapter_id paragraph_id sent_id token_id               
A Memory of Light 0          0            0       0         Non-Cosmere  
                                                  1         Non-Cosmere  
                                                  2         Non-Cosmere  
                                                  3         Non-Cosmere  
                                                  4         Non-Cosmere

## Sentiment on VOCAB

Use the Sentiment values from syuzhet

In [12]:
VOCAB = TOKENS['term_str'].value_counts().to_frame('n')
VOCAB.index.name = 'term_str'
VOCAB_SENT = VOCAB.join(SALEX, how='inner')
VOCAB_SENT

,n,syu_sentiment
term_str,,
like,6499,0.50
well,2992,0.80
right,2787,0.80
good,2309,0.75
enough,1997,-0.25
...,...,...
shipwreck,1,-0.50
conceited,1,-0.50
dubiously,1,-0.80


## VOCAB Sentiment at the BOW level

In [15]:
BOW = TOKENS.groupby(PARA+['term_str']).term_str.count().to_frame('n')
BOW_SENT = BOW.join(VOCAB_SENT['syu_sentiment'], on='term_str', how='inner')

for col in VOCAB_SENT.columns:
    BOW_SENT[col] = BOW_SENT[col] * BOW_SENT['n']
BOW_SENT

n  \
title                          chapter_id paragraph_id term_str       
A Memory of Light              0          0            birth      1   
                                                       forgotten  1   
                                                       mist       1   
                                                       myth       4   
                                          1            desolate   1   
...                                                              ..   
Yumi and the Nightmare Painter 0          42           kiss       1   
                                                       real       1   
                                                       share      1   
                                          43           worth      4   
                                          44           kiss       1   

                                                                  syu_sentiment  
title                          chapter_id paragraph_id term_str                  
A Memory of Light              0          0            birth               0.60  
                                                       forgotten          -0.50  
                                                       mist               -0.25  
                                                       myth               -1.00  
                                          1            desolate           -0.50  
...                                                                         ...  
Yumi and the Nightmare Painter 0          42           kiss                0.50  
                                                       real                0.25  
                                                       share               0.50  
                                          43           worth               3.00  
                                          44           kiss                0.50  

[223441 rows x 2 columns]

## Documnet Sentiment for Each Chapter

In [20]:
DOC_SENT = BOW_SENT.groupby(CHAP)[['syu_sentiment']].sum()
DOC_SENT

syu_sentiment
title                          chapter_id               
A Memory of Light              0                   14.10
                               1                   10.15
                               2                   22.00
                               3                  -16.85
                               4                   19.65
...                                                  ...
Words of Radiance              4                   29.20
                               5                    2.05
                               6                  -38.15
                               7                    9.15
Yumi and the Nightmare Painter 0                   28.05

[649 rows x 1 columns]

## Plot the Syuszhet Sentiment overtime narrative time

In [22]:
plot_df = DOC_SENT.reset_index()
plot_df['max_chapter'] = plot_df.groupby('title')['chapter_id'].transform(max)
plot_df['book_progress'] = plot_df['chapter_id'] / plot_df['max_chapter'].replace(0, 1)

books_to_plot = ['The Wheel of Time', 'The Way of Kings', 'The Final Empire']

plot_df['smoothed_sentiment'] = plot_df.groupby('title')['syu_sentiment'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())

subset_df  = plot_df[plot_df['title'].isin(books_to_plot)].copy()

fig = px.line(
    subset_df,
    x='book_progress',
    y='smoothed_sentiment',
    color='title',
    title='Sentiment Over Normalized Narrative Time',
    labels={
        'book_progress': 'Normalized Narrative Time',
        'smoothed_sentiment': 'Sentiment (Smoothed)',
        'title': 'Title'
    }
)
fig.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.5)
fig.update_layout(
    xaxis_title='Normalized Narrative Time',
    yaxis_title='Sentiment (Smoothed)',
    legend_title='Title',
    xaxis_tickformat='.2%'
)
fig.show()

In [40]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

plot_df = DOC_SENT.reset_index()
plot_df = plot_df.merge(LIB[['Series']], on='title', how='left')
plot_df['max_chapter'] = plot_df.groupby('title')['chapter_id'].transform(max)
plot_df['book_progress'] = plot_df['chapter_id'] / plot_df['max_chapter'].replace(0, 1)
plot_df['smoothed_sentiment'] = plot_df.groupby('title')['syu_sentiment'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())

series_to_plot = ['The Wheel of Time', 'The Stormlight Archive', 'Mistborn']
fig = make_subplots(
    rows=len(series_to_plot),
    cols=1,
    subplot_titles=series_to_plot,
    vertical_spacing=0.1
)
for i, series in enumerate(series_to_plot, start=1):
    series_data = plot_df[plot_df['Series'] == series]
    books_in_series = series_data['title'].unique()
    legend_ref = f"legend{i}" if i >1 else "legend"

    for book in books_in_series:
        book_data = series_data[series_data['title'] == book]
        fig.add_trace(
            go.Scatter(
                x=book_data['book_progress'],
                y=book_data['smoothed_sentiment'],
                mode='lines',
                name=book,
                legend=legend_ref
            ),
            row=i,
            col=1
        
        )
fig.update_layout(
    height=900,
    title='Syuzhet Sentiment Over Normalized Narrative Time by Series',
    legend=dict(y=0.85, yanchor='middle', x=1.02, title_text="The Wheel of Time"),
    legend2=dict(y=0.5, yanchor='middle', x=1.02, title_text="The Stormlight Archive"),
    legend3=dict(y=0.15, yanchor='middle', x=1.02, title_text="Mistborn")
)
fig.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.5)
fig.update_xaxes(tickformat='.2%', title_text='Normalized Narrative Time', row=3, col=1)
fig.update_yaxes(title_text='Sentiment (Smoothed)', row=2, col=1)
fig.show()
fig.write_image(directory_path + "/Output/Syuzhet_Series_Plot.png", scale=2)